# 🔬 GIADA roadmap — Task 12 + Task 13
Un solo notebook, due report separati. **Task 12:** corrente Ca-HVA analitica dai gate predetti dal modello Task 11 congelato, senza riaddestramento. **Task 13:** allineamento temporale della corrente autentica NEURON sul tracciato Task 7b. Il supplemento 11c resta rinviato e non entra nel confronto. Campo di validità: Ca-HVA+pas a un compartimento, non neurone completo.


In [ ]:
from pathlib import Path
import base64, json, os, shutil, subprocess, sys
from IPython.display import Javascript, display
WORK=Path('/kaggle/working/giada_roadmap_task12_13');REPO=WORK/'giada'
assert not WORK.exists(),f'Workspace già presente: {WORK}. Usa una sessione pulita.'
WORK.mkdir(parents=True)
subprocess.run(['git','clone','https://github.com/Zagred47/giada.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'fetch','origin','codex/surrogate-validity-audit'],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach','FETCH_HEAD'],check=True)
REVISION=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
assert (REPO/'src/giada_teacher/roadmap_current_contract.py').is_file(),'Revisione GIADA non contiene le Task 12/13.'
sys.path.insert(0,str(REPO))
for name in [n for n in list(sys.modules) if n=='src' or n.startswith('src.')]:del sys.modules[name]
from src.giada_teacher.roadmap_current_contract import verified_task11_checkpoints,evaluate_task12,evaluate_task13
print({'revision':REVISION})


## 📁 Input piccoli, verificati
Aggiungi `giada_roadmap_task11_causal_operator.zip` e `giada_cahva_active_closed_loop_microcanary.zip` (o i rispettivi Dataset Kaggle estratti). Si verificano hash del freeze e di tutti i checkpoint Task 11, e hash delle tracce native Task 7b. Non serve il dataset HayFlow da 6 GiB. Se Kaggle ha rinominato la sorgente, imposta `GIADA_TASK11_ARTIFACT` / `GIADA_TASK7B_ARTIFACT` ai percorsi ZIP o cartella.


In [ ]:
INPUT=Path('/kaggle/input')
def find_source(env,needles,verifier):
 override=os.environ.get(env)
 candidates=[Path(override).expanduser()] if override else []
 if INPUT.is_dir():
  candidates += [p for p in INPUT.rglob('*.zip') if any(n in str(p).lower() for n in needles)]
  candidates += [p.parent for p in INPUT.rglob('selection_freeze.json') if any(n in str(p).lower() for n in needles)]
  candidates += [p.parent for p in INPUT.rglob('trajectories.npz') if any(n in str(p).lower() for n in needles)]
 seen=set()
 for path in candidates:
  if str(path) in seen or not path.exists():continue
  seen.add(str(path))
  try:verifier(path);return path.resolve()
  except (ValueError,KeyError,FileNotFoundError,OSError):continue
 raise AssertionError(f'{env}: artefatto esatto non trovato. Aggiungi il Dataset o imposta la variabile al file ZIP/cartella.')
from src.giada_teacher.cahva_boundary_semantics_reassessment import verified_task7b_trace_bytes
TASK11=find_source('GIADA_TASK11_ARTIFACT',('task11','task-11','causal-operator','causal_operator'),verified_task11_checkpoints)
TASK7B=find_source('GIADA_TASK7B_ARTIFACT',('active-closed-loop','active_closed_loop'),verified_task7b_trace_bytes)
print({'task11':str(TASK11),'task7b':str(TASK7B)})


## 12 — ⚡ Corrente analitica dai gate predetti
Usiamo `path_full` e `effect_full` già selezionati dalla Task 11. Un ruolo indipendente, seed 12059, viene generato una sola volta; nessuna scelta sul suo esito. Confrontiamo la corrente totale predetta e due ibridi diagnostici (errore dei soli gate / del solo voltaggio), sul totale e sul quartile di corrente autentica non nulla più intenso. Unità mA/cm². Questa è una conferma sul riferimento sintetico Ca-HVA+pas validato contro NEURON, non una misura di generalizzazione sul neurone intero.


In [ ]:
import torch
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
OUTPUT=Path('/kaggle/working/artifacts/giada_roadmap_task12_13_current_contract')
assert not OUTPUT.exists(),f'Output già presente: {OUTPUT}. Sessione pulita necessaria.'
OUTPUT.mkdir(parents=True)
print('[GIADA Task 12] Modelli congelati, generazione ruolo indipendente...',flush=True)
task12=evaluate_task12(TASK11,device=DEVICE)
(OUTPUT/'task12_current_report.json').write_text(json.dumps(task12,indent=2),encoding='utf-8')
display({'valid':task12['valid'],'device':DEVICE,'seed':task12['independent_role_seed'],'windows':task12['window_count'],'signal_rms':task12['teacher_current_rms_ma_cm2'],'arms':task12['arms']})
assert task12['valid'] and not task12['training_performed'] and not task12['model_selection_performed']


## 13 — 🕒 Semantica temporale autentica
Sulle tracce native `.025 ms` confrontiamo quattro assegnazioni pre/post di V e gate rispetto a `ica` misurata. Escludiamo `gbar=0` dal voto perché è un controllo nullo. La conduttanza non era registrata direttamente: il report la etichetta esplicitamente come **inferita** da corrente e driving force pre-step. Il test non viene estrapolato a CVode o al neurone completo.


In [ ]:
print('[GIADA Task 13] Allineamento sulle tracce native verificate...',flush=True)
task13=evaluate_task13(TASK7B)
(OUTPUT/'task13_timing_report.json').write_text(json.dumps(task13,indent=2),encoding='utf-8')
final={'schema_version':'giada-roadmap-task12-13-v1','valid':bool(task12['valid'] and task13['valid']),'code_revision':REVISION,'task12_valid':task12['valid'],'task13_valid':task13['valid'],'task11c_status':'future_supplement_not_run','scope':'Ca_HVA+pas one compartment; Task13 native fixed-step .025 ms'}
(OUTPUT/'final_report.json').write_text(json.dumps(final,indent=2),encoding='utf-8')
display({'valid':task13['valid'],'native_episodes_nonzero_gbar':task13['native_episode_count'],'winner':task13['winner'],'current_active_samples':task13['current_active_samples'],'conductance_caveat':task13['conductance_caveat']})
assert final['valid']


## 📦 Download
ZIP piccolo con i due report distinti. Metodo Blob/base64 concordato per Kaggle.


In [ ]:
archive=Path(shutil.make_archive('/kaggle/working/giada_roadmap_task12_13_current_contract','zip',OUTPUT.parent,OUTPUT.name))
payload=base64.b64encode(archive.read_bytes()).decode('ascii')
display(Javascript(f"""const b=atob('{payload}');const a=new Uint8Array(b.length);for(let i=0;i<b.length;i++)a[i]=b.charCodeAt(i);const u=URL.createObjectURL(new Blob([a],{{type:'application/zip'}}));const l=document.createElement('a');l.href=u;l.download='{archive.name}';document.body.appendChild(l);l.click();l.remove();setTimeout(()=>URL.revokeObjectURL(u),1000);"""))
print({'archive':archive.name,'size_mib':round(archive.stat().st_size/2**20,2)})
